# Convection–Diffusion PINN V2.3.7
## Unified four-sampler protocol
MC-R, LHS-R, RAD-IID, and ALHS-R branch from one exact shared prefix per budget–seed.

In [ ]:
from pathlib import Path
import subprocess, sys, json, pandas as pd
CODE_DIR = Path.cwd().resolve()
OUTPUT_ROOT = CODE_DIR / "pde_v237_outputs"
RUN_PRESET = "smoke"  # smoke -> pilot -> confirm; do not use full yet
METHODS = "all"
BUDGETS = None  # pilot default: 250,500,1000
SEEDS = None    # pilot default: 0-2
DEVICE = "auto"
THREADS = 8
print("CODE_DIR:", CODE_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## 1. Verification
This does not train a research model. It checks the unified context and uniform q=1 weighting.

In [ ]:
subprocess.run([sys.executable, str(CODE_DIR/"verify_convection_diffusion_v237.py")], check=True)


## 2. Run selected preset
Start with `smoke`. After all four runs complete, change only `RUN_PRESET = "pilot"`.

In [ ]:
cmd=[sys.executable, str(CODE_DIR/"convection_diffusion_unified_v237.py"), "--preset", RUN_PRESET, "--output-root", str(OUTPUT_ROOT), "--methods", METHODS, "--device", DEVICE, "--threads", str(THREADS)]
if BUDGETS is not None: cmd += ["--budgets", str(BUDGETS)]
if SEEDS is not None: cmd += ["--seeds", str(SEEDS)]
print(" ".join(cmd))
process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout: print(line,end="")
rc=process.wait()
if rc: raise RuntimeError(f"Process failed with exit {rc}")


## 3. Inspect integrity and summary


In [ ]:
stage = RUN_PRESET
folder = OUTPUT_ROOT / stage
print((folder/"integrity.json").read_text() if (folder/"integrity.json").exists() else "No integrity report yet")
if (folder/"summary.csv").exists(): display(pd.read_csv(folder/"summary.csv"))


## 4. Export compact analysis summary


In [ ]:
folder=OUTPUT_ROOT/RUN_PRESET
parts=[]
for name in ["integrity.json","manifest.json","results_unified.csv","summary.csv","paired_differences.csv"]:
    p=folder/name
    if p.exists(): parts.append(f"\n==== {name} ====\n"+p.read_text(encoding="utf-8"))
target=OUTPUT_ROOT/f"V237_{RUN_PRESET}_ringkasan_untuk_analisis.txt"
target.write_text("\n".join(parts),encoding="utf-8")
print(target)
